# Fake News Detection Model Training

## Project Title
Algorithmic Fact-Verification Networks

## Objective
This notebook trains a simple machine learning model to classify news articles as Real or Fake using a Fake News Dataset.

## Model Approach
We use TF-IDF to convert text into numbers, then train a Logistic Regression model for binary classification.

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

In [4]:
df = pd.read_csv("../data/fake_news_dataset/train.csv")

In [5]:
df.head()

,index,title,text,subject,date,class,Unnamed: 6
0,0,PRESIDENT TRUMP Explains New “America First” R...,That s what we re talking about! Another campa...,politics,"Aug 2, 2017",Fake,NaN
1,1,TERMINALLY ILL FORMER MISS WI: “Until my last ...,How is it that Sean Hannity is the only media ...,politics,"Oct 4, 2016",Fake,NaN
2,2,Cruz Humiliated By Moderator After Lie About ...,Almost immediately after learning that longtim...,News,"February 13, 2016",Fake,NaN
3,3,"Russia revels in Trump victory, looks to sanct...",MOSCOW (Reuters) - For all their mutual praise...,politicsNews,"November 9, 2016",Real,NaN
4,4,Trump's bid to open U.S. monuments to developm...,WASHINGTON (Reuters) - The Trump administratio...,politicsNews,"May 26, 2017",Real,NaN


In [6]:
df.columns

Index(['index', 'title', 'text', 'subject', 'date', 'class', 'Unnamed: 6'], dtype='object')

In [7]:
df["class"].value_counts()

class
Fake                20886
Real                19113
February 5, 2017        1
Name: count, dtype: int64

## clean the useless column

In [8]:
df = df.drop(columns=["Unnamed: 6"], errors="ignore")
df.head()

,index,title,text,subject,date,class
0,0,PRESIDENT TRUMP Explains New “America First” R...,That s what we re talking about! Another campa...,politics,"Aug 2, 2017",Fake
1,1,TERMINALLY ILL FORMER MISS WI: “Until my last ...,How is it that Sean Hannity is the only media ...,politics,"Oct 4, 2016",Fake
2,2,Cruz Humiliated By Moderator After Lie About ...,Almost immediately after learning that longtim...,News,"February 13, 2016",Fake
3,3,"Russia revels in Trump victory, looks to sanct...",MOSCOW (Reuters) - For all their mutual praise...,politicsNews,"November 9, 2016",Real
4,4,Trump's bid to open U.S. monuments to developm...,WASHINGTON (Reuters) - The Trump administratio...,politicsNews,"May 26, 2017",Real


In [9]:
df["class"].value_counts(dropna=False)

class
Fake                20886
Real                19113
February 5, 2017        1
Name: count, dtype: int64

In [10]:
df["class"] = df["class"].astype(str).str.strip()

df["label"] = df["class"].map({"Real": 0,"Fake": 1})

df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

print(df["label"].isna().sum())

0


In [12]:
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")

In [13]:
df["label"] = df["class"].map({"Real": 0,"Fake": 1})
df[["content", "class", "label"]].head()

,content,class,label
0,PRESIDENT TRUMP Explains New “America First” R...,Fake,1
1,TERMINALLY ILL FORMER MISS WI: “Until my last ...,Fake,1
2,Cruz Humiliated By Moderator After Lie About ...,Fake,1
3,"Russia revels in Trump victory, looks to sanct...",Real,0
4,Trump's bid to open U.S. monuments to developm...,Real,0


In [14]:
df["label"].value_counts()

label
1    20886
0    19113
Name: count, dtype: int64

## Selecting Features and Labels

In supervised machine learning, the dataset is divided into:

- Features (X): The input data used for learning.
- Labels (y): The correct answers the model tries to predict.

For this project:

- X = Combined news content
- y = Fake (1) or Real (0)

In [15]:
X = df["content"]
y = df["label"]

In [16]:
print("First feature:\n")
print(X.iloc[0])

print("\nLabel:")
print(y.iloc[0])

First feature:

PRESIDENT TRUMP Explains New “America First” RAISE Act…No More Welfare For New Immigrants, Migrants…Favors English Speaking Immigrants…Protects Jobs For Minorities, US Workers From Being Replaced And MORE! [VIDEO] That s what we re talking about! Another campaign promise kept. No wonder the Democrats and their media allies fear President Trump. When is the last time a politician actually followed through on a promise they made to the American voters that helped them to get elected?President Trump joined two Republican senators on Wednesday to champion legislation overhauling legal immigration in America, calling for a merit-based system that would significantly cut admissions over the next decade.Speaking at the White House, the president called it  the most significant reform to our immigration system in a half century. As a candidate, I campaigned on creating a merit-based immigration system that protects U.S. workers and taxpayers, and that is why we are here today, 

## Splitting the Dataset

The dataset is divided into two parts:

- Training set (80%) used for learning.
- Testing set (20%) used for evaluating the model.

This helps determine how well the model performs on unseen data.

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 31999
Testing samples: 8000


## Text Vectorization using TF-IDF

Machine learning models cannot process raw text directly.

TF-IDF converts news articles into numerical feature vectors while assigning greater importance to informative words and reducing the influence of very common words.

In [19]:
vectorizer = TfidfVectorizer(stop_words="english")

In [20]:
X_train_vectorized = vectorizer.fit_transform(X_train)

In [21]:
X_test_vectorized = vectorizer.transform(X_test)

In [22]:
print(X_train_vectorized.shape)

(31999, 106676)


## Model Training using Logistic Regression

The Logistic Regression algorithm is trained using the TF-IDF feature vectors from the training dataset.

During training, the algorithm learns the relationship between the numerical features and their corresponding labels (Real or Fake News).

In [23]:
model = LogisticRegression(max_iter=1000)

In [24]:
model.fit(X_train_vectorized, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Model Evaluation

After training, the model is tested using unseen test data. This helps determine how well the model can classify new articles as Real or Fake.

## Make predictions

In [25]:
y_pred = model.predict(X_test_vectorized)

## check Accuracy

In [26]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Model Accuracy (%):", accuracy * 100)

Model Accuracy: 0.987875
Model Accuracy (%): 98.7875


## Classification report

In [27]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      3849
           1       0.99      0.99      0.99      4151

    accuracy                           0.99      8000
   macro avg       0.99      0.99      0.99      8000
weighted avg       0.99      0.99      0.99      8000



## Confusion matrix

In [28]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[3811   38]
 [  59 4092]]


## Testing the Model with Custom News

This section demonstrates how the trained model predicts whether a new news article is Real or Fake.

In [46]:
news = "WASHINGTON (Reuters) - The U.S. government announced new economic measures on Monday following discussions with lawmakers and financial regulators. Officials said the measures are intended to support businesses and stabilize markets."

In [47]:
news_vector = vectorizer.transform([news])

## Predict

In [48]:
prediction = model.predict(news_vector)
probability = model.predict_proba(news_vector)

print("Prediction:", prediction[0])
print("Probability Real:", probability[0][0])
print("Probability Fake:", probability[0][1])

if prediction[0] == 0:
    print("🟢 Prediction: Real News")
else:
    print("🔴 Prediction: Fake News")

Prediction: 0
Probability Real: 0.9980380195600043
Probability Fake: 0.001961980439995781
🟢 Prediction: Real News


## Save Model

In [49]:
joblib.dump(model, "../models/fake_news_model.pkl")

['../models/fake_news_model.pkl']